<a href="https://colab.research.google.com/github/dansojo/Super-Resolution/blob/main/RCAN_%2B_ESPCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt



In [ ]:
# Residual Block for RCAN
class ResidualBlock(nn.Module):
  def __init__(self, num_channels):
    super(ResidualBlock, self).__init__()
    self.conv1 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
    self.relu = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)

  def forward(self, x):
    residual = x
    x = self.relu(self.conv1(x))
    x = self.conv2(x)
    return x + residual

# Residual Channel Attention Block
class ChannelAttention(nn.Module):
  def __init__(self, num_channels, reduction_ratio=16):
    super(ChannelAttention, self).__init__()
    self.avg_pool = nn.AdaptiveAvgPool2d(1)
    self.fc = nn.Sequential(
        nn.Conv2d(num_channels, num_channels // reduction_ratio, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(num_channels // reduction_ratio, num_channels, kernel_size=1),
        nn.Sigmoid()
    )

  def forward(self, x):
    y = self.avg_pool(x)
    y = self.fc(y)
    return x * y

In [ ]:
# RCAN + ESPCN Model


# ESPCN (Efficient Sub-Pixel Convolutional Network) for upsampling
class ESPCN(nn.Module):
    def __init__(self, scale_factor=2):
        super(ESPCN, self).__init__()
        self.conv = nn.Conv2d(64, 64 * (scale_factor ** 2), kernel_size=3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(scale_factor)

    def forward(self, x):
        x = self.conv(x)
        return self.pixel_shuffle(x)

# RCAN with ESPCN for upsampling
class RCAN_ESPCN(nn.Module):
    def __init__(self, num_channels=64, num_blocks=20, num_groups=10, scale_factor=2):
        super(RCAN_ESPCN, self).__init__()
        self.conv1 = nn.Conv2d(3, num_channels, kernel_size=3, padding=1)

        self.residual_groups = nn.Sequential(
            *[self._make_group(num_channels, num_blocks) for _ in range(num_groups)]
        )

        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        self.upsample = ESPCN(scale_factor)

    def _make_group(self, num_channels, num_blocks):
        layers = [ResidualBlock(num_channels) for _ in range(num_blocks)]
        layers.append(ChannelAttention(num_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.residual_groups(x)
        x = self.conv2(x)
        x = self.upsample(x)
        return x


In [ ]:
# Dataset class for SET5
class SET5Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.lr_images = []
        self.hr_images = []

        # 저해상도(LR) 이미지 파일을 image_LR 폴더에서 불러옴
        for img_name in os.listdir(os.path.join(root_dir, 'image_LR')):
            self.lr_images.append(os.path.join(root_dir, 'image_LR', img_name))

        # 고해상도(HR) 이미지 파일을 image_HR 폴더에서 불러옴
        for img_name in os.listdir(os.path.join(root_dir, 'image_HR')):
            self.hr_images.append(os.path.join(root_dir, 'image_HR', img_name))

        # 이미지 정렬
        self.lr_images = sorted(self.lr_images)
        self.hr_images = sorted(self.hr_images)

    def __len__(self):
        return len(self.lr_images)

    def __getitem__(self, idx):

        # 이미지를 RGB로 변환하여 불러오기
        lr_image = Image.open(self.lr_images[idx]).convert("RGB")
        hr_image = Image.open(self.hr_images[idx]).convert("RGB")

        if self.transform:
            lr_image = self.transform(lr_image)
            hr_image = self.transform(hr_image)

        return lr_image, hr_image

In [ ]:
# Training function
def train_and_save_model(model, model_name, dataset, num_epochs=50, lr=1e-4, batch_size=16):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for lr_images, hr_images in dataloader:
            lr_images, hr_images = lr_images.cuda(), hr_images.cuda()  # GPU 사용 가정
            optimizer.zero_grad()
            sr_images = model(lr_images)
            loss = criterion(sr_images, hr_images)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()


        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(dataloader):.4f}")

    torch.save(model.state_dict(), f"/content/drive/MyDrive/Super-Resolution/{model_name}.pth")
    print(f"Model {model_name} saved.")

In [ ]:
# 간단한 Evaluation function (PSNR, SSIM 계산)
def evaluate_model(model, dataset):
    model.eval()
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    total_psnr = 0
    total_ssim = 0
    count = 0

    with torch.no_grad():
        for lr_image, hr_image in dataloader:
            lr_image, hr_image = lr_image.cuda(), hr_image.cuda()  # GPU 사용 가정
            sr_image = model(lr_image).cpu().squeeze(0).permute(1, 2, 0).numpy()
            hr_image = hr_image.cpu().squeeze(0).permute(1, 2, 0).numpy()

            # 이미지 값 [0, 1]로 클리핑
            sr_image = np.clip(sr_image, 0, 1)
            hr_image = np.clip(hr_image, 0, 1)

            # PSNR, SSIM 계산
            psnr_value = psnr(hr_image, sr_image)
            ssim_value = ssim(hr_image, sr_image, multichannel=True)

            total_psnr += psnr_value
            total_ssim += ssim_value
            count += 1

    avg_psnr = total_psnr / count
    avg_ssim = total_ssim / count

    # 최종 성능 출력
    print(f"Average PSNR: {avg_psnr:.4f}, Average SSIM: {avg_ssim:.4f}")

    return avg_psnr, avg_ssim

In [ ]:
# Main script
if __name__ == "__main__":
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  # Transform for the dataset (only normalization to [0, 1] range)
  transform = transforms.Compose([
      transforms.Resize((128, 128)),
      transforms.ToTensor()  # Convert image to tensor and normalize to [0, 1]
  ])

  # Load SET5 dataset
  dataset = SET5Dataset(root_dir='/content/drive/MyDrive/Super-Resolution/Set5', transform=transform)

  # Initialize RCAN_ESPCN model
  model = RCAN_ESPCN().cuda()  # Assuming GPU is available

  # Train the model and save it
  train_and_save_model(model, "RCAN_ESPCN", dataset)

  # Evaluate the model using the test dataset
  avg_psnr, avg_ssim = evaluate_model(model, dataset)

  print(f"Final Average PSNR: {avg_psnr:.4f}, Average SSIM: {avg_ssim:.4f}")